# Prompts 101: Understanding the Basics

Before optimizing prompts, you need the building blocks: tokens, pricing, quotas, and latency. These fundamentals determine how much you pay, how fast responses arrive, and how much capacity you can use — and every later optimization is measured against them.

## Learning Objectives

By the end of this notebook, you will be able to:
- Explain tokens, count input tokens before generation, and read observed usage
- Calculate inference costs based on token usage, and see how savings compound at scale
- Understand TPM/RPM quotas and the output burndown rate, and their impact on throughput
- Measure visible TTFT and E2E latency, and explain how TPS and TTLT differ
- Use the Converse API on `bedrock-runtime`, and recognize when to reach for the Bedrock Mantle endpoint

## Why This Matters

At production scale, small inefficiencies compound dramatically:
- **1,000 extra tokens per request × 1 million requests = 1 billion unnecessary tokens processed** — real money
- Token economics is the foundation for every cost optimization that follows
- Right-sizing output limits and understanding quotas helps manage throughput without truncating useful answers

The default workhorse is **Sonnet 5 through Converse on bedrock-runtime**. Prices and capabilities use the shared catalog dated **2026-09-24**. Live inference cells incur charges; optional comparisons are labelled.


## Setup

Select **Bedrock Workshop (Python 3.13)** and run the cells in order. The workshop environment and shared resources are prepared. If you need the local kernel commands, follow [Jupyter Notebook 101](00-jupyter-notebook-101.ipynb).

The next cells load the SDKs, shared model registry, and pricing helpers. Later, [Observability and Evaluation](02-observability-and-evaluation.ipynb) connects these measurements to request traces in CloudWatch or Langfuse.


In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Select Bedrock Workshop (Python 3.13), then restart the kernel.")

REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "workshop_utils").is_dir()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook inside the workshop repository.")

print("Kernel:", sys.executable)
print("Python:", sys.version.split()[0])
print({name: version(name) for name in ("boto3", "botocore", "bedrock-agentcore")})
import workshop_utils

print("Shared package:", workshop_utils.__file__)


## 1. Choose the model and define the task

A fair experiment keeps the model and settings fixed while you change the prompt. Otherwise, a cheaper answer might come from switching models rather than improving instructions. The **experiment manifest** is the record of the exact model/profile, Region, API, effort, prompt version, dataset, and pricing date.

The setup cell selects the workhorse and small-model roles in `us-east-1`. It can make small billed checks before selection: Sonnet 5 is preferred, Sonnet 4.6 is the same-provider fallback when unavailable, and Haiku 4.5 fills the small role. Read the selected IDs in the output; the subsequent comparison keeps them fixed. These setup checks are separate from the application measurements below.

An **inference profile** is the routing identity used for the model request. A `global.` profile can route across Regions; a geographic profile has a narrower routing scope. The exact profile matters for both supported features and price.

For supported models, **Global CRIS offers approximately 10% lower input and output prices** than geographic CRIS. Where in-Region and geographic prices match, the same saving applies against in-Region inference. Broader routing is the tradeoff; compare the selected model’s prices and your routing requirements.


In [ ]:
import json
import os
import time

import boto3
import pandas as pd
from botocore.config import Config
from dotenv import load_dotenv
from IPython.display import display

from workshop_utils.bedrock import NormalizedUsage, build_converse_request, normalize_usage
from workshop_utils.models import Support, UnsupportedFeatureError, resolve_model
from workshop_utils.pacing import attach_boto3, get_workshop_pacer
from workshop_utils.preflight import preflight_models
from workshop_utils.pricing import calculate_cost, get_price

load_dotenv(REPO / ".env", override=False)
REGION = "us-east-1"
# Explicit setup I/O: STS identity plus small billable probes on cache misses.
# Probe attempts are setup overhead and are not baseline usage or latency.
REFRESH_PREFLIGHT = False  # After permission changes, restart and set True for a new experiment.
PREFLIGHT_CACHE = REPO / ".models.json"
model_override = os.getenv("WORKSHOP_MODEL_ID") or None
probe_runtime = boto3.client(
    "bedrock-runtime", region_name=REGION,
    config=Config(connect_timeout=5, read_timeout=90, retries={"total_max_attempts": 1}),
)
attach_boto3(probe_runtime, get_workshop_pacer())
probe_sts = boto3.client(
    "sts", region_name=REGION,
    config=Config(connect_timeout=5, read_timeout=10, retries={"total_max_attempts": 1}),
)
try:
    PREFLIGHT = preflight_models(
        probe_runtime, sts_client=probe_sts, region=REGION,
        overrides={"workhorse": model_override} if model_override else None,
        cache_path=PREFLIGHT_CACHE, refresh=REFRESH_PREFLIGHT,
    )
finally:
    probe_runtime.close()
    probe_sts.close()

SELECTED = PREFLIGHT.models["workhorse"]
MODEL_ID = SELECTED.model_id
EFFORT = "low" if SELECTED.capabilities.support("effort") is Support.SUPPORTED else None
PREFLIGHT_REPORT = PREFLIGHT.as_dict()
display(PREFLIGHT_REPORT)
print("New probe attempts (separate setup overhead):",
      sum(not item["cache_hit"] for item in PREFLIGHT_REPORT["probes"]))
print("Probe cache:", PREFLIGHT_CACHE)

PRICE = get_price(SELECTED)
EXPERIMENT = {
    **SELECTED.as_dict(), "effort": EFFORT, "service_tier": "standard",
    "prompt_version": "fundamentals-v1", "dataset_version": "fundamentals-smoke-v1",
    "pricing_date": PRICE.verified_on, "price_evidence": PRICE.evidence,
    "tools": [], "knowledge_base": None, "setup_preflight": PREFLIGHT_REPORT,
}
display(EXPERIMENT)
# Create the application client after preflight; baseline timing starts at its call cell.
runtime = boto3.client(
    "bedrock-runtime", region_name=REGION,
    config=Config(connect_timeout=5, read_timeout=90, retries={"total_max_attempts": 1}),
)
attach_boto3(runtime, get_workshop_pacer())


### Prompts are task specifications

Make four parts explicit: **context**, the **task**, the **inputs**, and the **success criteria**. Put stable instructions and context ahead of changing user input. A useful output constraint specifies the audience, content, length, and what to do when the information is insufficient.

Compare “Explain returns” with the prompt below. The second prompt supplies evidence and states what a successful answer contains. It asks for the answer and supporting facts, rather than a transcript of private reasoning. On supported reasoning models, use effort to control reasoning depth; response length still needs its own instruction.

In our example, the evidence is: “TechMart accepts headphone returns within 30 days with a receipt and original packaging.” The customer bought headphones 14 days ago and has a receipt. A useful answer confirms the time and receipt conditions, and asks about packaging. “Yes, return them” would miss a condition even though it sounds helpful.

Read `POLICY`, `QUESTION`, and `SYSTEM` in the next cell. Notice how stable policy goes in the system instruction while the customer's question goes in a user message. The request is displayed before it is sent, letting you inspect what the model will actually receive.


In [ ]:
POLICY = "TechMart accepts headphone returns within 30 days with a receipt and original packaging."
QUESTION = "I bought headphones 14 days ago and have a receipt. Can I return them?"
SYSTEM = (
    "You are a TechMart support assistant. Answer using only the policy. "
    "Use at most two sentences and include any condition the customer has not confirmed. "
    f"Policy: {POLICY}"
)
request = build_converse_request(
    SELECTED, [{"role": "user", "content": [{"text": QUESTION}]}],
    system=[{"text": SYSTEM}], max_tokens=512, effort=EFFORT,
)
# Inspect the request before sending it: no temperature or provider-unknown fields.
display(request)


---

## 2. Understanding Tokens

Tokens are the fundamental units that LLMs use to process text. Understanding tokenization is crucial because:
- **Pricing** is based on token count, not character count
- **Context limits** are measured in tokens
- **Quota limits** (TPM) are based on tokens processed

### What is a Token?

A token can be:
- A complete word (for example, "hello")
- Part of a word ("tokenization" might be 2-3 tokens)
- Punctuation or whitespace
- Special characters

**Rule of thumb**: For English text, 1 token is approximately 4 characters or 0.75 words.

### CountTokens API

Amazon Bedrock offers **free input token counting** before generation: `CountTokens` on `bedrock-runtime` for supported models, and Anthropic’s `count_tokens` API on `bedrock-mantle` for newer Claude models such as Sonnet 5. This is useful for:

- Estimating costs before processing
- Ensuring prompts fit within context limits
- Checking minimum token requirements for caching

The four-characters-per-token rule is an approximation for English, not a billing measurement. Vocabulary, language, punctuation, and formatting change the ratio. Message framing, system prompts, and tool definitions also contribute input tokens.

**Try it:** the next cell counts the system instruction and question you defined above. For Sonnet 5, it uses Mantle’s counting endpoint; the following answer is still generated through Converse. Treat the count as a planning estimate because API formatting can differ, then inspect the actual usage returned by inference. Counting is free; generating the answer is billed.

Source: [Count tokens before inference](https://docs.aws.amazon.com/bedrock/latest/userguide/count-tokens.html).


In [ ]:
counted_input_tokens = None
if SELECTED.capabilities.support("count_tokens") is Support.SUPPORTED:
    # Count the same Converse input without generating an answer.
    body = {"messages": request["messages"], "system": request["system"]}
    counted_input_tokens = runtime.count_tokens(
        modelId=SELECTED.base_model_id, input={"converse": body}
    )["inputTokens"]
elif SELECTED.base_model_id == "anthropic.claude-sonnet-5":
    from anthropic import AnthropicBedrockMantle, PermissionDeniedError

    # Retry only the first-use activation status, and only this counting request.
    COUNT_ACTIVATION_WAIT_SECONDS, COUNT_ACTIVATION_POLL_SECONDS = 180, 10
    with AnthropicBedrockMantle(aws_region=REGION, timeout=30.0, max_retries=0) as client:
        count_deadline = time.monotonic() + COUNT_ACTIVATION_WAIT_SECONDS
        while True:
            remaining = count_deadline - time.monotonic()
            if remaining <= 0:
                raise TimeoutError(
                    "Token counting is not ready: first-use model activation is still in progress. "
                    f"The {COUNT_ACTIVATION_WAIT_SECONDS}-second wait ended. "
                    "Wait a few minutes, then rerun only this token-count cell. "
                    "Activation may take longer than this wait limit."
                ) from None
            try:
                # Keep the same bare model ID, system instruction and question.
                count = client.messages.count_tokens(
                    model=SELECTED.base_model_id,
                    system=SYSTEM,
                    messages=[{"role": "user", "content": QUESTION}],
                    timeout=min(30.0, remaining),
                )
            except PermissionDeniedError as exc:
                error = exc.body
                if isinstance(error, dict):
                    error = error.get("error", error)
                message = error.get("message") if isinstance(error, dict) else error
                if (exc.status_code != 403 or not isinstance(message, str)
                        or message.partition(".")[0] != "Your subscription to the model is being set up"):
                    raise  # Other permission failures are not activation retries.
                remaining = max(0, count_deadline - time.monotonic())
                if remaining:
                    delay = min(COUNT_ACTIVATION_POLL_SECONDS, remaining)
                    print(f"First-use model activation is in progress; retrying token counting "
                          f"in {delay:g}s ({remaining:.0f}s left in this wait).", flush=True)
                    time.sleep(delay)
            else:
                counted_input_tokens = count.input_tokens
                print("Token counting is ready.")
                break
else:
    print("No counting example for this selection; inspect the next response's usage.")

if counted_input_tokens is not None:
    print("Input tokens before generation:", counted_input_tokens)


### Send the request and read the response

The next cell makes **one billed Converse request** using the prompt you inspected. First read the answer as a customer would: does it mention the missing packaging condition? Then read the three measurements:

- **Stop reason:** whether the model finished its turn or reached the output limit.
- **Usage:** how many input and output tokens the service processed.
- **Client end-to-end time:** how long this notebook waited for the complete response.

A short answer is valuable when it remains complete. A response cut off at `max_tokens` needs a different interpretation from a naturally concise answer. The code keeps the complete assistant message for possible multi-turn reuse, while displaying its text.


In [ ]:
started = time.perf_counter()
result = runtime.converse(**request)
client_e2e_ms = (time.perf_counter() - started) * 1000
answer = "".join(block["text"] for block in result["output"]["message"]["content"] if "text" in block)
print(answer)
print("Stop reason:", result["stopReason"])
print("Raw usage:", json.dumps(result["usage"], indent=2))
print("Client end-to-end ms:", round(client_e2e_ms, 1))


## 3. Understanding Pricing

Amazon Bedrock's on-demand inference pricing is based on tokens. Input is the context you send; output is what the model generates, including billed reasoning where present. Output often has a higher per-token price, which is why a focused answer can matter as much as a shorter prompt.

For a request without caching:

**cost = (input tokens × input price + output tokens × output price) / 1,000,000**, when prices are quoted per million tokens.

Caching introduces separately priced input meters. Each token should be counted in the appropriate meter once:

| Converse meter | Meaning | Price to use |
|---|---|---|
| `inputTokens` | Uncached input | Regular input rate |
| `cacheReadInputTokens` | Input reused from cache | Cache read rate |
| `cacheWriteInputTokens` | Input written to cache | Write rate for that TTL |
| `outputTokens` | Generated output, including billed reasoning | Output rate |
| `cacheDetails` | Breakdown of writes by TTL when supplied | Detail of writes, not another charge |

The table below compares **USD per million tokens, Standard tier**, with the date each price was last checked. Confirm current rates on the Bedrock pricing page before budgeting. The response-cost cell then applies those rates to your actual usage; a missing rate or unidentified cache-write TTL leaves the estimate unknown.


In [ ]:
TABLE_MODELS = [
    "global.anthropic.claude-haiku-4-5-20251001-v1:0",
    "global.anthropic.claude-sonnet-4-6", "global.anthropic.claude-sonnet-5",
    "global.anthropic.claude-opus-4-8", "global.anthropic.claude-opus-5",
    "global.openai.gpt-5.6-luna", "global.openai.gpt-5.6-sol",
]
price_rows = []
for identifier in TABLE_MODELS:
    price = get_price(identifier, allow_inferred=True)
    price_rows.append({
        "model/profile": identifier, "input $/M": price.input_per_million,
        "output $/M": price.output_per_million, "cache read $/M": price.cache_read_per_million,
        "cache writes $/M": dict(price.cache_write_per_million),
        "evidence": price.evidence, "price date": price.verified_on,
    })
display(pd.DataFrame(price_rows))


For the selected global profiles, the notebook’s price table lists Sonnet 5 at **$2/$10**, Haiku 4.5 at **$1/$5**, and Opus 5 at **$5/$25** per million input/output tokens. Opus 5 cache reads cost **0.10×** regular input ($0.50/M). Preserve any inferred label on a non-global SKU-to-profile mapping.

GPT-5.6 Luna documents a 30-minute cache policy on **Responses**, with writes at 1.25× input and reads at 0.1×. Its global short-context rate is $0.20 input/$1.20 output per million tokens. Above 272K input tokens, the whole-request input/output multipliers are 2×/1.5×; price each request before aggregation. Do not transfer Luna's prices or capability rules to Sol.

Sources: [Bedrock pricing](https://aws.amazon.com/bedrock/pricing/), [Claude pricing](https://platform.claude.com/docs/en/about-claude/pricing), [GPT-5.6 Luna card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-openai-gpt-56-luna.html).


In [ ]:
from workshop_utils.pricing import AmbiguousCacheUsageError, InferredPriceError, UnknownPriceError

observed_usage = normalize_usage(result["usage"], source="converse")
try:
    observed_cost = calculate_cost(SELECTED, observed_usage)
except (AmbiguousCacheUsageError, InferredPriceError, UnknownPriceError) as exc:
    observed_cost = None
    print("Cost unknown:", exc)
else:
    display(observed_cost.as_dict())
# A cache write without TTL evidence remains unpriced; do not assume the cheaper TTL.


### Interactive Cost Calculator and ROI at Different Scales

Let's see how savings compound at different request volumes. Imagine removing duplicated context from a 5,000-token prompt while still producing the same 150-token answer. The new prompt has 3,000 input tokens: a 40% input reduction.

Before running the calculator, predict whether the **total bill** also falls by 40%. It will fall by less, because the output charge has not changed. Read the monthly table from 100 to 100,000 requests per day to see why a small per-request improvement becomes worth pursuing.

This **synthetic accounting exercise makes no inference calls**. Compare 5,000 versus 3,000 uncached input tokens while holding output at 150 tokens. This isolates the arithmetic of prompt shortening; it does not establish that a real shorter prompt preserves answer quality. Validate that separately on held-out scenarios.


In [ ]:
def cost_for_tokens(input_tokens, output_tokens):
    return calculate_cost(SELECTED, NormalizedUsage(input_tokens, output_tokens)).total_cost

baseline_cost = cost_for_tokens(5000, 150)
candidate_cost = cost_for_tokens(3000, 150)
savings = baseline_cost - candidate_cost
print(f"Baseline: ${baseline_cost:.6f}; candidate: ${candidate_cost:.6f}")
print(f"Input reduction: 40%; total cost reduction: {100 * savings / baseline_cost:.2f}%")
display(pd.DataFrame([
    {"requests/day": volume, "baseline $/30 days": baseline_cost * volume * 30,
     "candidate $/30 days": candidate_cost * volume * 30, "savings $/30 days": savings * volume * 30}
    for volume in (100, 1000, 10000, 100000)
]))


### Why cost per solved task matters

A cheaper request can fail more often, trigger retries, or delegate additional model/tool work. Compare **the cost of all attempts and model calls divided by the number of successfully solved tasks**, using an explicit success criterion. If no task succeeds, the metric is undefined—not zero. Keep judge and workshop infrastructure charges separate.

The figures above assume one attempt and equal output lengths. Treat them as estimates until measured usage and held-out quality support the comparison.


## 4. Prompt caching: the minimum belongs to the exact model

**Prompt caching stores a processed prefix so later requests can reuse it.** Think of a support assistant that receives the same product manual with every new customer question. Reprocessing the manual each time costs input tokens and time; reuse can reduce that work while leaving the question and answer fresh.

The first eligible use may pay a cache-write premium. Later hits pay the lower read rate until the cache expires. Caching is therefore most useful for sizeable, stable context reused within the cache's lifetime, rather than a one-off short question.

For explicit Claude caching, request order is **tools → system → messages**. Put stable policy/documents before the checkpoint and the changing question after it. A cache point is a separate content block:

```python
system = [
    {"text": stable_policy_document},
    {"cachePoint": {"type": "default", "ttl": "5m"}},
]
messages = [{"role": "user", "content": [{"text": changing_question}]}]
```

The short policy in this notebook is below the default model's minimum, so adding a checkpoint would not demonstrate caching. Use the Playbook's eligible document fixture and inspect returned counters. Character count cannot guarantee eligibility.

**Sonnet 5: 1,024 tokens; Opus 5/5.5: 512; Opus 4.8: 1,024; Haiku 4.5: 4,096.** Claude supports at most four checkpoints and 5m/1h TTLs in these catalog entries. Longer TTL checkpoints must come first when mixing TTLs. A single trailing checkpoint can search earlier block boundaries (approximately 20 blocks), but neither explicit nor best-effort implicit caching guarantees a hit.

GPT's Converse path does not accept Claude `cachePoint` blocks. Implicit caching observations depend on the exact generation, prompt, API and date. Explicit GPT prefix controls such as `prompt_cache_breakpoint` belong to a separately selected Responses API experiment.

Source: [Bedrock prompt caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).


In [ ]:
capability_rows = []
for identifier in TABLE_MODELS:
    model = resolve_model(identifier, region=REGION)
    capability_rows.append({
        "model/profile": identifier,
        "Converse cache mode": model.capabilities.cache_mode,
        "min prefix tokens": model.capabilities.cache_min_tokens,
        "TTLs": ", ".join(model.capabilities.cache_ttls),
        "cachePoint": model.capabilities.support("cache_points").value,
        "output quota factor": model.capabilities.output_quota_factor,
    })
display(pd.DataFrame(capability_rows))


Check **both** cache write and read counters. A successful response with zero cache reads does not prove caching works. Changing a stable prefix, model, effort/thinking settings, tool definitions, or some multimodal content can change cache eligibility. Cross-region routing and time between calls also matter.

For a 5-minute Claude write costing 1.25× and subsequent reads costing 0.1×, a second use can recover the write premium; realized savings depend on how much of each request is cacheable and whether it hits. A 1-hour write costs more up front, so choose TTL from actual reuse intervals rather than defaulting to the longest option.


## 5. Latency Impact

Token count affects responsiveness as well as cost. A customer waiting for an answer experiences two phases: waiting for text to begin and waiting for the answer to finish. Streaming exposes the first phase instead of hiding everything behind a single completed response.

| Metric | What it measures | Why it matters |
|---|---|---|
| **Visible TTFT** — time to first token | Request start to the first nonempty text delta | How soon the user sees an answer begin |
| **TPS** — tokens per second | Output generation rate over a defined interval | How quickly text arrives once it starts |
| **TTLT** — time to last token | Request start to the final generated token | When the generated answer is complete |
| **Client E2E** — end-to-end latency | Request start until the client finishes receiving the response | The wait observed by this application |
| **Provider latency** | Service-reported `metrics.latencyMs`, where supplied | The service's view of request timing |

The next cell uses **ConverseStream** to timestamp the first visible text and completion. A non-streaming call returns the whole answer at once, so it cannot measure TTFT. This helper reports TTFT and E2E; it does not claim a separate TTLT or TPS measurement. Reasoning can consume time and output tokens before visible text, so dividing all billed output by a visible-text interval would mislabel TPS.

Run the two-sentence example and compare TTFT with E2E. Streaming can make a response feel quicker even if total completion time is unchanged. Repeat comparable requests before drawing a performance conclusion: network conditions, cache state, and model work all affect one sample.


In [ ]:
def measure_latency(prompt, *, max_tokens=512):
    stream_request = build_converse_request(
        SELECTED, [{"role": "user", "content": [{"text": prompt}]}],
        max_tokens=max_tokens, effort=EFFORT,
    )
    start = time.perf_counter()
    first_text = None
    text_parts = []
    usage = None
    provider_ms = None
    stop_reason = None
    response = runtime.converse_stream(**stream_request)
    stream = response["stream"]
    try:
        for event in stream:
            failure = next((key for key in event if key.endswith("Exception")), None)
            if failure:
                raise RuntimeError(f"ConverseStream failed: {failure}: {event[failure]}")
            text = event.get("contentBlockDelta", {}).get("delta", {}).get("text", "")
            if text:
                if first_text is None:
                    first_text = time.perf_counter()
                text_parts.append(text)
            if "metadata" in event:
                usage = event["metadata"].get("usage")
                provider_ms = event["metadata"].get("metrics", {}).get("latencyMs")
            if "messageStop" in event:
                stop_reason = event["messageStop"].get("stopReason")
    finally:
        stream.close()
    end = time.perf_counter()
    return {
        "text": "".join(text_parts), "stop_reason": stop_reason,
        "visible_ttft_ms": None if first_text is None else (first_text - start) * 1000,
        "e2e_ms": (end - start) * 1000, "provider_ms": provider_ms, "usage": usage,
        "model_id": MODEL_ID, "endpoint": SELECTED.endpoint, "effort": EFFORT,
    }

# Reference: https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html
CACHING_REFERENCE = (
    "Amazon Bedrock prompt caching reuses previously computed state for an unchanged prompt prefix. "
    "A cache hit reduces repeated input processing; the model still generates a new response. "
    "It does not replay a saved answer."
)

def caching_explanation_prompt(length):
    return (
        f"Using only the reference below, explain Bedrock prompt caching in {length}. "
        "State what is reused and whether output generation still runs.\n"
        f"Reference: {CACHING_REFERENCE}"
    )

# One billed streaming request when you run this cell.
latency_sample = measure_latency(caching_explanation_prompt("two short sentences"))
display(latency_sample)


Read the streamed answer against the [Bedrock prompt-caching reference](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html): it must describe reuse of prefix computation and fresh output generation, without claiming that saved answers are replayed. Inspect the returned text; a successful stream alone does not establish correctness.

**Exercise:** predict which changes affect visible TTFT, total time, or both. A smaller static prefix can reduce prefill work; a shorter requested answer can reduce decoding; a cache hit can reduce input processing; effort can change reasoning work. Validate each prediction instead of assuming shorter text always means proportionally lower latency.

The next optional comparison makes two additional requests. It intentionally changes output requirements, so it illustrates a latency tradeoff—not equal-quality optimization. Both use the same output cap, with room for the longer answer and reasoning. Check that both finish with `end_turn` before comparing their complete responses.


In [ ]:
RUN_LATENCY_COMPARISON = False
if RUN_LATENCY_COMPARISON:
    samples = [
        measure_latency(caching_explanation_prompt("one sentence"), max_tokens=1024),
        measure_latency(caching_explanation_prompt("six numbered paragraphs"), max_tokens=1024),
    ]
    display(pd.DataFrame(samples))
    for label, sample in zip(("One sentence", "Six numbered paragraphs"), samples):
        print(f"\n{label}:\n{sample['text']}")  # Read the full answer, beyond the table preview.
else:
    print("Optional comparison skipped; no extra inference calls.")


## 6. Bedrock API Endpoints

**Converse** provides a common request and response shape for supported models: messages have roles and content blocks, and responses include an assistant message and usage. **ConverseStream** is its streaming counterpart. Your application supplies conversation history on each call; the API does not remember a chat for you.

**InvokeModel** accepts a model-specific request body on the runtime endpoint. It is useful for a supported provider-native feature that the unified interface does not expose. Model portability still requires checking capabilities: changing an ID alone does not make every setting valid.

An **endpoint** is where the request goes; an **API** defines its body, response and features. Do not count Mantle as a method alongside Converse and InvokeModel.

| Endpoint | API shapes relevant here | Model/routing choice | Important boundary |
|---|---|---|---|
| `bedrock-runtime` | Converse / ConverseStream; InvokeModel; model-supported Messages / Responses / Chat Completions | Exact runtime IDs; Sonnet 5 uses a geo/global inference profile | Feature support still varies by model **and API** |
| `bedrock-mantle` | Anthropic Messages; OpenAI-compatible Responses / Chat Completions, where supported | Provider-native model names, regional service | No runtime CRIS routing; different capabilities, quotas and accounting |

Start with bedrock-runtime for new applications when it supports the features you need. For Sonnet 5, bare on-demand runtime IDs are not supported; use a catalogued inference profile. Single-region Mantle use is a separate choice. Runtime integrations such as Guardrails are not automatically present on every API: for GPT, Responses is not the Converse Guardrails path.

Sources: [Sonnet 5 card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5.html), [API compatibility](https://docs.aws.amazon.com/bedrock/latest/userguide/models-api-compatibility.html), [Responses API](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-responses-api.html).


Both endpoints support stored Responses conversations and provider SDK integration on eligible models. Mantle adds server-side tools, background requests, Projects, and Workspaces. Choose those features explicitly; reasoning effort alone does not require changing endpoints. [Endpoint comparison](https://docs.aws.amazon.com/bedrock/latest/userguide/endpoints.html).

### Parameter validation protects the comparison

Sonnet 5 and Opus 5 reject sampling parameters such as `temperature` on the selected Converse surface. Effort uses `additionalModelRequestFields.output_config.effort` for Claude, and `additionalModelRequestFields.reasoning.effort` for supported GPT models. Haiku 4.5 has no effort parameter; extended thinking instead uses an explicit token budget. Check sampling support for the selected model rather than inferring it from the model family.

Sonnet 5 and Opus 5 can disable thinking on their supported paths. Opus 5 defaults to adaptive thinking with high effort; check its capability record before using xhigh or max. Claude 5 does not inherit Haiku 4.5's structured-output or forced-tool settings. Never copy these controls across a provider boundary or silently discard a rejected setting.

The request builder enforces its dated capability catalog before an inference call. The next cell is offline.


In [ ]:
if SELECTED.capabilities.support("temperature") is Support.UNSUPPORTED:
    try:
        build_converse_request(SELECTED, request["messages"], temperature=0)
    except UnsupportedFeatureError as exc:
        print("Expected validation failure:", exc)
else:
    print("This selected model does not have the Sonnet 5 temperature restriction.")

gpt_comparison = resolve_model("global.openai.gpt-5.6-luna", region=REGION)
gpt_request = build_converse_request(
    gpt_comparison, [{"role": "user", "content": [{"text": "Reply in one sentence."}]}],
    max_tokens=512, effort="low",
)
print("Separate GPT experiment; request constructed only, not invoked:")
display(gpt_request)


### Optional Mantle example: a separate experiment

**Why use a provider-native API?** An existing application may already use Anthropic Messages or an OpenAI-compatible interface. Both Bedrock endpoints offer supported paths for those request shapes. This example uses Mantle, with its own model identifiers and quota accounting.

Inspect the function below even if you leave the optional call disabled: `messages.create()` returns the provider's content/usage object, whereas Converse returned Bedrock's `output.message` envelope. Comparing those shapes explains the API choice; the endpoint's name alone does not predict quality or latency.

This optional exercise calls **Sonnet 5 on Mantle** through `AnthropicBedrockMantle`, using the supplied `anthropic[bedrock]` dependency and AWS authentication. Enable the exercise flag to run this separate API comparison.

If you run it, record Mantle's model ID, region, SDK, response usage, and pricing evidence separately. Do not bill it with the `global.` runtime price merely because the model name is similar. A rejected request remains an error for this surface; do not silently route it back to Converse.

For OpenAI Responses comparisons, also choose storage/retention explicitly (`store=false` when persistence is unnecessary). Do not assume its storage, native tools, or Guardrails match Converse or OpenAI's direct service.

Source: [Claude in Bedrock](https://platform.claude.com/docs/en/build-with-claude/claude-in-amazon-bedrock).


In [ ]:
def mantle_example(prompt):
    from anthropic import AnthropicBedrockMantle

    from workshop_utils.pacing import get_workshop_pacer

    client = AnthropicBedrockMantle(aws_region=REGION, timeout=90.0, max_retries=0)
    response = get_workshop_pacer().call(
        client.messages.create,
        model="anthropic.claude-sonnet-5", max_tokens=512,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "endpoint": "bedrock-mantle", "api": "messages", "region": REGION,
        "model_id": "anthropic.claude-sonnet-5", "effort": "low",
        "text": "".join(block.text for block in response.content if block.type == "text"),
        "usage": response.usage.model_dump(),
    }

RUN_MANTLE_COMPARISON = False
if RUN_MANTLE_COMPARISON:
    display(mantle_example(caching_explanation_prompt("two short sentences")))
else:
    print("Mantle comparison skipped; no Mantle request.")


## 7. TPM and RPM Quotas

Bedrock applies throughput limits so applications can share service capacity. **RPM (requests per minute)** limits the number of calls; **TPM (tokens per minute)** limits token consumption for the selected model/profile/Region. A burst of many tiny requests can hit RPM; fewer long requests can hit TPM.

Quota accounting has two stages on the selected runtime path:

1. **Before generation**, the service reserves approximately input tokens plus the maximum output allowance.
2. **After generation**, consumption is based on uncached input + cache writes + (actual output × the model's output burndown factor). Cache reads do not count toward this final consumption.

This explains why `max_tokens` should match the task. Reserving space for a huge answer on a short classification request can reduce concurrency. A cap that is too small can truncate the answer and trigger retries instead.

| Model | Runtime output burndown in the dated catalog |
|---|---:|
| Sonnet 5, Opus 5 | 10× |
| Opus 4.8 | 15× |
| Sonnet 4.6, Haiku 4.5 | 5× |
| GPT-5.6 Luna | 10× |

The example below reserves **3,000 + 2,000 = 5,000** tokens, then uses 120 output tokens. With a 10× factor, final consumption is **3,000 + 120 × 10 = 4,200**, returning 800 tokens of reserved capacity. More output could instead require additional capacity.

**The burndown factor changes quota consumption, never the dollar bill.** The calculator prints both to make the distinction visible. Mantle has separate input/output accounting, so this runtime arithmetic does not carry over to that experiment.

Source: [Token burndown](https://docs.aws.amazon.com/bedrock/latest/userguide/quotas-token-burndown.html).


In [ ]:
# Synthetic, no-cache quota example; no AWS calls or account quota assumptions.
factor = SELECTED.capabilities.output_quota_factor
if factor is None:
    print("Quota example skipped: no verified output factor for this exact model.")
else:
    input_tokens, output_limit, actual_output = 3000, 2000, 120
    reservation = input_tokens + output_limit
    final_consumption = input_tokens + actual_output * factor
    display({
        "model": MODEL_ID, "output quota factor": factor,
        "initial reservation": reservation, "final consumption": final_consumption,
        "net tokens returned (negative means additional consumption)": reservation - final_consumption,
        "token bill USD (no quota multiplier)": cost_for_tokens(input_tokens, actual_output),
    })


## Apply the foundations

You have followed one request from task definition to measured answer: structured its instructions, inspected token usage, estimated cost, measured streaming latency, and worked through quota accounting.

Before moving on, explain these results in your own words: why a 40% input reduction produces a smaller total cost reduction; why streaming can feel faster; why cached reads have a separate price; and why a quota multiplier does not multiply the bill.

Next, [Observability and Evaluation](02-observability-and-evaluation.ipynb) turns individual measurements into traces and checks the answer against a reference. The [Optimization Playbook](../02-optimization-playbook/01-low-effort.ipynb) then applies model selection, clearer prompts, caching, and other techniques. For each change, keep the task and success criteria fixed so “cheaper” still means “solves the problem.”

## References

[Count tokens before inference](https://docs.aws.amazon.com/bedrock/latest/userguide/count-tokens.html), [Global CRIS benefits](https://docs.aws.amazon.com/bedrock/latest/userguide/global-cross-region-inference.html), [Bedrock pricing](https://aws.amazon.com/bedrock/pricing/), [Prompt caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html), [Converse API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html), [Service quotas](https://docs.aws.amazon.com/bedrock/latest/userguide/quotas.html).
